# Jets: CPEN / CAPEN forward–backward stability

Graph classification (TopTagging, directed $k$NN). The coupled-network / $\mu$P theory says hidden states $Z_t^{(\ell)}$ (particle stream $t=X$, edge stream $t=E$) stay $\Theta(1)$ in

$$
\rho_t^{(\ell)}=\frac{\|Z_t^{(\ell)}\|_F^2}{N_t D},
$$

and that a **width-transferred** Adam step $\eta=\eta_0/\sqrt{D}$ moves them by an amount that does not blow up as $D$ (and $L$) grow. $\Delta\rho$ is the same coordinate on $\Delta Z = Z_{\mathrm{after}}-Z_{\mathrm{init}}$.

$N_t$ is the number of **valid** tokens in that stream (unpadded particles, nonempty edges). The batch is treated as one token pool.

The loss is the training graph-level CE: both families read out $(z_X+z_E)/\sqrt{2}$ with $z_X,z_E\in\mathbb{R}^{B\times C}$. $T$ Adam steps are **on each minibatch**, not a full epoch.

**Layout.** Each family is a $3\times 3$ grid vs $\#$ trainable parameters:

- **Rows:** encoder $\ell=0$; first and last residual ($\ell=1$ open markers, $\ell=L$ filled); decoder $\ell=L{+}1$.
- **Columns:** init; after 1 Adam step; after $T=1000$ Adam steps.
- Particle traces are **blue circles**; edge traces are **orange squares**.
- Solid $=\rho$, dashed $=\Delta\rho$. Init has $\rho$ only.

Every point is a mean over several independent jet minibatches, with error bars and a $\pm 1$ std band.

**On $\ell=L{+}1$.** The decoder acts *after* pooling (CPEN $\alpha$-pools with the energy weights, CAPEN mask-means), so this layer is not a per-token state. We plot
$$
\rho_t^{(L+1)}=\frac{\|z_t\|_F^2}{B\,C}.
$$
The probe is only kept when $(z_X+z_E)/\sqrt{2}$ reproduces the model's own logits. That pooled $\rho$ sits well below the hidden states, so every panel is log–log. Under $\mu$P the init logits are *expected* to shrink with width while the one-step logit move stays $O(1)$.

Helpers: `cpen.training.stability`.


In [9]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import torch

REPO = Path("../..").resolve() if (Path("../..") / "src" / "cpen").exists() else Path("..").resolve()
sys.path.insert(0, str(REPO / "src"))

import cpen.training.stability as stability

importlib.reload(stability)

from cpen.training.stability import (
    DEFAULT_ARCHS,
    DEFAULT_FIG_DIR,
    DEFAULT_N_ADAM_STEPS,
    DEFAULT_N_BATCHES,
    cuda_mem_line,
    load_jets_knn_batches,
    plot_layerwise_msq,
    plot_stability_vs_params,
    release_cuda,
    run_stability_ladder,
)
from cpen.training.transfer_plots import setup_mpl_style

setup_mpl_style()
FIG_DIR = DEFAULT_FIG_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
release_cuda()
print("stability:", stability.__file__)
print("device:", DEVICE)
print("archs:", DEFAULT_ARCHS)
print("FIG_DIR:", FIG_DIR)
print("batches:", DEFAULT_N_BATCHES)
if DEVICE.type == "cuda":
    print("cuda:", cuda_mem_line())
    alloc = torch.cuda.memory_allocated() / 2**30
    if alloc > 2.0:
        print(
            f"WARNING: this kernel already holds {alloc:.2f} GiB. "
            "Kernel → Restart, then run from the top. "
            "Also kill leftover ipykernel PIDs in nvidia-smi from earlier OOM runs."
        )


stability: /home/jdezoort/coupled-particle-edge-networks/src/cpen/training/stability.py
device: cuda
archs: ((2, 128), (3, 192), (4, 256), (5, 384), (6, 512))
FIG_DIR: /home/jdezoort/coupled-particle-edge-networks/notebooks/figures
batches: 4
cuda: NVIDIA A100 80GB PCIe MIG 1g.10gb  allocated=0.03 GiB  used=0.57/9.50 GiB


## Batches of jets

Several TopTagging minibatches, graphs built on the fly (`8-NN`). Each architecture is trained independently on each batch so the error band is batch-to-batch scatter, not init noise. Model seed is shared; only the jet sample changes.


In [10]:
DATA_ROOT = Path("/scratch/gpfs/BHANIN/jdezoort/datasets/toptagging")
N_JETS = 16
NUM_PARTICLES = 128
K = 8
N_BATCHES = DEFAULT_N_BATCHES

batches = load_jets_knn_batches(
    DATA_ROOT,
    n_jets=N_JETS,
    n_batches=N_BATCHES,
    num_particles=NUM_PARTICLES,
    k=K,
    split="train",
    seed=0,
    device=DEVICE,
)
for i, batch in enumerate(batches):
    n_valid = int(batch["mask"].sum())
    n_edge = int(batch["incidence"].bool().any(dim=-1).sum())
    print(
        f"batch {i}: B={batch['x'].size(0)}  N={batch['x'].size(1)}  "
        f"M={batch['edge_x'].size(1)}  n0={batch['x'].size(-1)}  "
        f"m0={batch['edge_x'].size(-1)}  valid particles={n_valid}  "
        f"nonempty edges={n_edge}  labels={batch['y'].tolist()}"
    )


batch 0: B=16  N=128  M=1024  n0=7  m0=4  valid particles=715  nonempty edges=5720  labels=[0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0]
batch 1: B=16  N=128  M=1024  n0=7  m0=4  valid particles=746  nonempty edges=5968  labels=[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1]
batch 2: B=16  N=128  M=1024  n0=7  m0=4  valid particles=746  nonempty edges=5968  labels=[1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0]
batch 3: B=16  N=128  M=1024  n0=7  m0=4  valid particles=754  nonempty edges=6032  labels=[1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0]


## Ladder

- **CPEN**: transformer-like residuals, incidence operators, degree-norm, affine-free LN, dropout 0.
- **CAPEN**: same residual layout with support attention (this is the linear-vs-attention comparison). Set `FAMILIES = ("cpen", "capen-llama")` if you want the SwiGLU / all-to-all jets trunk instead.

Adam uses the training map $\eta=\eta_0/\sqrt{D}$ and $\varepsilon=10^{-14}$. The objective is graph CE on $(z_X+z_E)/\sqrt{2}$. $T=1000$ steps are **on that batch**, not a full epoch — we are probing the parameterization, not measuring tagging AUC.

After a CUDA OOM, **restart the kernel** before re-running (IPython keeps the failed frame, so the GPU stays full). Leftover `ipykernel` processes from earlier sessions will also pin a 10 GiB MIG — check `nvidia-smi`.


In [11]:
SMOKE = False  # True → 2 sizes, T=5, 2 batches; faster on login/CPU nodes

ETA_0 = 0.25
HEADS = 8  # CAPEN; ignored by CPEN. D=64L is always divisible by 8.
FAMILIES = ("cpen", "capen")
SEED = 0

if SMOKE:
    ARCHS = ((2, 128), (4, 256))
    N_ADAM_STEPS = 5
    batches = batches[:2]
else:
    ARCHS = DEFAULT_ARCHS
    N_ADAM_STEPS = DEFAULT_N_ADAM_STEPS

print(
    f"families={FAMILIES}  archs={ARCHS}  η0={ETA_0}  T={N_ADAM_STEPS}  "
    f"n_batches={len(batches)}  smoke={SMOKE}  obj=graph"
)
release_cuda()
if DEVICE.type == "cuda":
    print("cuda before ladder:", cuda_mem_line())

df = run_stability_ladder(
    batches,
    families=FAMILIES,
    archs=ARCHS,
    eta_0=ETA_0,
    n_adam_steps=N_ADAM_STEPS,
    heads=HEADS,
    seed=SEED,
    device=DEVICE,
    objective="graph",
)
last = df.loc[
    df.apply(lambda r: r["stage"] == f"layer{int(r['depth']) - 1}", axis=1)
]
display(
    last.pivot_table(
        index=["family", "depth", "width", "n_params"],
        columns="token",
        values=["msq_init", "dmsq_1", "msq_T", "dmsq_T"],
        aggfunc="mean",
    )
)


families=('cpen', 'capen')  archs=((2, 128), (3, 192), (4, 256), (5, 384), (6, 512))  η0=0.25  T=1000  n_batches=4  smoke=False  obj=graph
cuda before ladder: NVIDIA A100 80GB PCIe MIG 1g.10gb  allocated=0.04 GiB  used=0.57/9.50 GiB
[stability] NVIDIA A100 80GB PCIe MIG 1g.10gb  allocated=0.04 GiB  used=0.57/9.50 GiB
[stability] batch 1/4  cpen          L=2 D=128  T=1000  obj=graph
[stability] batch 1/4  cpen          L=3 D=192  T=1000  obj=graph
[stability] batch 1/4  cpen          L=4 D=256  T=1000  obj=graph
[stability] batch 1/4  cpen          L=5 D=384  T=1000  obj=graph
[stability] batch 1/4  cpen          L=6 D=512  T=1000  obj=graph
[stability] batch 1/4  capen         L=2 D=128  T=1000  obj=graph
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/jdezoort/.conda/envs/mamba-env/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 3508, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_2134204/3560169667.py", line 24, in <module>
    df = run_stability_ladder(
  File "/home/jdezoort/coupled-particle-edge-networks/src/cpen/training/stability.py", line 711, in run_stability_ladder
    run_one_architecture(
  File "/home/jdezoort/coupled-particle-edge-networks/src/cpen/training/stability.py", line 560, in run_one_architecture
    return _run_one_architecture_body(
  File "/home/jdezoort/coupled-particle-edge-networks/src/cpen/training/stability.py", line 613, in _run_one_architecture_body
    loss_t = _objective_step(model, batch, opt, objective=objective)
  File "/home/jdezoort/coupled-particle-edge-networks/src/cpen/training/stability.py", line 499, in _objective_step
    logits = model(batch["x"], batch["edge_x"], **_forward_kw

## $\rho$ and $\Delta\rho$ vs $\#$ parameters

If the theory is right:

- the encoder / residual init $\rho$ stays $O(1)$ across the ladder;
- the one-step $\Delta\rho$ does not grow with $D$;
- after $T$ steps, feature size is still $O(1)$ and the cumulative move stays controlled.

Particle = blue circles, edge = orange squares. Solid $\rho$, dashed $\Delta\rho$. On the residual row, open markers are $\ell=1$ and filled markers are $\ell=L$. Bands are mean $\pm$ 1 std over batches.


In [ ]:
plot_stability_vs_params(
    df,
    n_adam_steps=N_ADAM_STEPS,
    stem="jets_stability",
    fig_dir=FIG_DIR,
    show=True,
)


## Layerwise check at one size

Same $\rho$, now vs layer index ($\ell=0$ encoder, $\ell=1,\ldots,L$ residuals, $\ell=L{+}1$ decoder). Residual $\mu$P should keep this from exploding in depth. Error bars are again batch-to-batch.


In [ ]:
# Prefer (4, 256) when present; otherwise the largest architecture in the run.
_archs = list(df[["depth", "width"]].drop_duplicates().itertuples(index=False, name=None))
DEPTH, WIDTH = (4, 256) if (4, 256) in _archs else _archs[-1]
plot_layerwise_msq(
    df,
    depth=DEPTH,
    width=WIDTH,
    ycol="msq_init",
    stem=f"jets_stability_layers_L{DEPTH}_D{WIDTH}",
    fig_dir=FIG_DIR,
    show=True,
)
plot_layerwise_msq(
    df,
    depth=DEPTH,
    width=WIDTH,
    ycol="dmsq_1",
    stem=f"jets_stability_layers_dZ_L{DEPTH}_D{WIDTH}",
    fig_dir=FIG_DIR,
    show=True,
)
